# Benchmark grid: NMF-gene perturbation (3 × 3 × 3)

Generates the final benchmark dataset by sweeping all combinations of:

| Parameter | Values |
|-----------|--------|
| `PERTURB_FRAC` | 0.30, 0.50, 0.70 |
| `FIXED_LAM` | 0.5, 0.3, 0.15 |
| `TOP_GENES_PER_FACTOR` | 1, 5, 10 |

**3 × 3 × 3 = 27 h5ad files** saved to `benchmark/generated_benchmark_data_final/`.

**Setup (same puck as B.3):** bottom half of Puck 06 (y-flipped).

**Gene selection:**
- HVGs identified on a temporary log-normalised copy (normalize_total → log1p), then discarded.
- NMF (k=15) run on the raw counts subset to those HVGs, **unit-variance scaled without centering** (preserves non-negativity).
- For each factor, the top-N genes by H-matrix loading are collected; unique genes pooled across all factors form the perturbation set.

**Perturbation:** Poisson(λ) counts added to the selected genes inside a central circle (~20% of beads). Only a random fraction (`PERTURB_FRAC`) of circle beads are actually perturbed; `in_circle` (full circle) remains the ground truth.

**NMF is computed once; gene selection and perturbation loop over all 27 conditions.**

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc
from itertools import product
from pathlib import Path
from sklearn.decomposition import NMF

RAND_SEED  = 28
N_FACTORS  = 15

DATA_DIR = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/All_Pucks_h5ad')
OUT_DIR  = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/benchmark/generated_benchmark_data_final')
OUT_DIR.mkdir(exist_ok=True)
print(f'Output directory: {OUT_DIR}')

## Parameter grid

In [ ]:
PERTURB_FRACS       = [0.30, 0.50, 0.70]
FIXED_LAMS          = [0.5,  0.3,  0.15]
TOP_GENES_LIST      = [1, 5, 10]

grid = list(product(PERTURB_FRACS, FIXED_LAMS, TOP_GENES_LIST))
print(f'Total conditions: {len(grid)}')
print('perturb_frac | fixed_lam | top_genes')
for pf, lam, tg in grid:
    print(f'  {pf:.0%}          {lam}        {tg}')

## Load Puck 06 and define circle

In [ ]:
adata06 = ad.read_h5ad(DATA_DIR / 'Puck_Num_06.h5ad')
adata06 = adata06[adata06.obs['IsOutsideCCF'] == 0].copy()
print(adata06)

In [ ]:
y_all       = adata06.obs['Raw_Slideseq_Y'].values.astype(float)
split_point = y_all.min() + (y_all.max() - y_all.min()) / 2
print(f'Y range: {y_all.min():.0f} – {y_all.max():.0f},  split at {split_point:.1f}')

adata06_bot = adata06[y_all > split_point].copy()

# Flip Y so bottom half has same orientation as top
y_bot = adata06_bot.obs['Raw_Slideseq_Y'].values.astype(float)
adata06_bot.obs['Raw_Slideseq_Y'] = y_bot.max() + y_bot.min() - y_bot

print(f'Bottom half: {adata06_bot.n_obs:,} beads')

In [ ]:
x = adata06_bot.obs['Raw_Slideseq_X'].values.astype(float)
y = adata06_bot.obs['Raw_Slideseq_Y'].values.astype(float)

cx = (x.min() + x.max()) / 2
cy = (y.min() + y.max()) / 2
puck_radius   = max(x.max() - x.min(), y.max() - y.min()) / 3
circle_radius = puck_radius / np.sqrt(5)

dist      = np.sqrt((x - cx)**2 + (y - cy)**2)
in_circle = dist <= circle_radius
adata06_bot.obs['in_circle'] = in_circle

print(f'Puck center:   ({cx:.0f}, {cy:.0f})')
print(f'Circle radius: {circle_radius:.0f}  (~{in_circle.mean():.1%} of beads)')
print(f'Circle beads (ground truth): {in_circle.sum():,} / {len(in_circle):,}')

fig, ax = plt.subplots(figsize=(5, 5))
colors = np.where(in_circle, '#e84545', '#d3d3d3')
ax.scatter(x, y, c=colors, s=0.5, linewidths=0, edgecolors='none', rasterized=True)
ax.add_patch(plt.Circle((cx, cy), circle_radius, fill=False, edgecolor='red', linewidth=1.5))
ax.set_aspect('equal', 'box'); ax.axis('off')
ax.set_title(f'Puck 06 — bottom: circle selection ({in_circle.mean():.0%} of beads)')
plt.tight_layout(); plt.show()

## NMF decomposition (computed once)

In [ ]:
adata_nmf = adata06_bot.copy()

_tmp = adata_nmf.copy()
sc.pp.normalize_total(_tmp, target_sum=1e4)
sc.pp.log1p(_tmp)
sc.pp.highly_variable_genes(_tmp, n_top_genes=2000, flavor='seurat', subset=False)
adata_nmf.var['highly_variable'] = _tmp.var['highly_variable']
del _tmp

adata_nmf = adata_nmf[:, adata_nmf.var['highly_variable']].copy()
print(f'NMF input shape: {adata_nmf.shape}')

sc.pp.scale(adata_nmf, zero_center=False)

X_nmf_input = adata_nmf.X
if sp.issparse(X_nmf_input):
    X_nmf_input = X_nmf_input.toarray()
X_nmf_input = X_nmf_input.astype(np.float32)

nmf_model = NMF(
    n_components=N_FACTORS,
    init='nndsvda',
    random_state=RAND_SEED,
    max_iter=1000,
    solver='cd',
)
W = nmf_model.fit_transform(X_nmf_input)
H = nmf_model.components_

gene_names = (
    adata_nmf.var['features'].values
    if 'features' in adata_nmf.var.columns
    else adata_nmf.var_names.values
)

print(f'W: {W.shape},  H: {H.shape}')
print(f'Reconstruction error: {nmf_model.reconstruction_err_:.2f}')

## Helper functions

In [ ]:
def select_nmf_genes(H, gene_names, n_top, n_factors):
    """Return unique genes (ordered by first appearance) for top-n per factor."""
    seen, ordered = set(), []
    for fi in range(n_factors):
        for gi in np.argsort(H[fi])[::-1][:n_top]:
            g = gene_names[gi]
            if g not in seen:
                ordered.append(g)
                seen.add(g)
    return ordered


def perturb_puck(adata_base, nmf_genes_ordered, perturb_frac, fixed_lam, rand_seed=RAND_SEED):
    """Return a perturbed copy of adata_base."""
    adata_case = adata_base.copy()

    all_circle_idx = np.where(adata_case.obs['in_circle'].values)[0]

    if perturb_frac < 1.0:
        rng_frac   = np.random.default_rng(rand_seed + 1)
        circle_idx = rng_frac.choice(
            all_circle_idx,
            size=int(len(all_circle_idx) * perturb_frac),
            replace=False
        )
        circle_idx = np.sort(circle_idx)
    else:
        circle_idx = all_circle_idx

    feat_to_col       = {g: i for i, g in enumerate(adata_case.var['features'])}
    nmf_genes_present = [g for g in nmf_genes_ordered if g in feat_to_col]
    gene_cols         = np.array([feat_to_col[g] for g in nmf_genes_present])

    X_sub = adata_case.X[np.ix_(circle_idx, gene_cols)]
    if sp.issparse(X_sub):
        X_sub = X_sub.toarray()
    X_sub = X_sub.astype(np.float32)

    rng       = np.random.default_rng(rand_seed)
    additions = rng.poisson(fixed_lam, size=X_sub.shape).astype(np.float32)

    row_idx = np.repeat(circle_idx, len(gene_cols))
    col_idx = np.tile(gene_cols, len(circle_idx))
    delta   = sp.csr_matrix(
        (additions.flatten(), (row_idx, col_idx)),
        shape=adata_case.X.shape, dtype=np.float32
    )
    adata_case.X = (adata_case.X + delta).tocsr()

    return adata_case, len(nmf_genes_present), len(circle_idx)


def clean_for_save(adata):
    a = adata.copy()
    a.raw = None
    if '_index' in a.var.columns:
        a.var = a.var.rename(columns={'_index': 'gene_id'})
    if a.var.index.name == '_index':
        a.var.index.name = None
    return a


print('Helper functions defined.')

## Generate all 27 benchmark files

In [ ]:
saved_files = []

for perturb_frac, fixed_lam, top_genes in grid:
    tag = f'frac{perturb_frac:.0%}_lam{fixed_lam}_top{top_genes}genes'
    out_path = OUT_DIR / f'adata06_bot_case_nmfGenes_{tag}.h5ad'

    # Gene selection for this top_genes value
    nmf_genes = select_nmf_genes(H, gene_names, n_top=top_genes, n_factors=N_FACTORS)

    # Perturbation
    adata_case, n_genes_present, n_perturbed_beads = perturb_puck(
        adata06_bot, nmf_genes, perturb_frac, fixed_lam
    )

    # Save
    clean_for_save(adata_case).write_h5ad(out_path)

    size_mb = out_path.stat().st_size / 1e6
    print(
        f'[{tag}]  genes={n_genes_present}  '
        f'perturbed_beads={n_perturbed_beads:,}  '
        f'saved {size_mb:.0f} MB'
    )
    saved_files.append(str(out_path))

print(f'\nDone. {len(saved_files)} files saved to {OUT_DIR}')

## Save NMF gene lists (one CSV per top-genes value)

In [ ]:
for top_genes in TOP_GENES_LIST:
    nmf_genes = select_nmf_genes(H, gene_names, n_top=top_genes, n_factors=N_FACTORS)

    records = []
    for fi in range(N_FACTORS):
        top_idx = np.argsort(H[fi])[::-1][:top_genes]
        for rank, gi in enumerate(top_idx):
            records.append({
                'gene':    gene_names[gi],
                'factor':  fi + 1,
                'rank':    rank + 1,
                'loading': float(H[fi, gi]),
            })

    df = pd.DataFrame(records)
    csv_path = OUT_DIR / f'nmf_genes_top{top_genes}_per_factor.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved {csv_path.name}  ({len(df)} rows, {len(nmf_genes)} unique genes)')

## Summary of generated files

In [ ]:
print(f'Files in {OUT_DIR}:\n')
for f in sorted(OUT_DIR.glob('*.h5ad')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)')
print()
for f in sorted(OUT_DIR.glob('*.csv')):
    print(f'  {f.name}')

# Summary table
rows = []
for pf, lam, tg in grid:
    tag  = f'frac{pf:.0%}_lam{lam}_top{tg}genes'
    path = OUT_DIR / f'adata06_bot_case_nmfGenes_{tag}.h5ad'
    rows.append({'perturb_frac': pf, 'fixed_lam': lam, 'top_genes': tg,
                 'file': path.name, 'size_MB': round(path.stat().st_size / 1e6)})

summary_df = pd.DataFrame(rows)
print()
print(summary_df.to_string(index=False))